# SMS-Spam Collection Dataset

Our necessary libraries:
sci-kit learn,
pandas,
nltk

In [ ]:
!pip install pandas scikit-learn nltk

## 1.- Preprocessing
Functionalities for our libraries:

In [ ]:
import pandas as pd
import numpy as np
import re

#that converts text to numbers
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

#for data splitting
from sklearn.model_selection import train_test_split

#the models:
#logistic regression is the base classifier, the one that provides probability estimates for its predictions
from sklearn.linear_model import LogisticRegression
#self training uses the base classifier (in this case logistic regression) to label the unlabeled data with high confidence and iteratively retrains the model using the pseudo-labeled examples
from sklearn.semi_supervised import SelfTrainingClassifier

#for evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#text processing NLTK
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download("averaged_perceptron_tagger_eng")



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

# Loading the raw dataset

In [ ]:

raw_url = "https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"

#to read the csv
df = pd.read_csv(raw_url, encoding="latin-1")

#we rename the columns to make them clearer
df = df.rename(columns={"v1": "label", "v2": "text"})

#we keep only the useful columns because the dataset contain extra empty columns...
df = df[["label", "text"]]

#quick preview of the dataset
print(df.shape)
df.head()

(5572, 2)


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
#we check the class distribution to see if the dataset is balanced or imbalanced
df["label"].value_counts()


,count
label,
ham,4825
spam,747


# Text preprocessing

In [ ]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

def nltk_pos_to_wordnet(tag):

    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def preprocess_text(text):
    #lower case
    text = text.lower()

    #removing URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    #removing numbers, punctuation
    text = re.sub(r"[^a-z\s]", " ", text)

    #whitespaces
    text = re.sub(r"\s+", " ", text).strip()

    #tokenization
    tokens = word_tokenize(text)

    #removing stopwords and very short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]

    #POS tagging
    tagged_tokens = pos_tag(tokens)

    #lemmatization
    lemmas = [
        lemmatizer.lemmatize(word, pos=nltk_pos_to_wordnet(tag))
        for word, tag in tagged_tokens
    ]


    return " ".join(lemmas)

#we apply preprocessing to all docs
df["clean_text"] = df["text"].apply(preprocess_text)

df[["label", "text", "clean_text"]].head()


,label,text,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...",jurong point crazy available bugis great world...
1,ham,Ok lar... Joking wif u oni...,lar joking wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win cup final tkts may te...
3,ham,U dun say so early hor... U c already then say...,dun say early hor already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though


# Text representation

In [ ]:
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X_tfidf.shape)
print(X_tfidf)

TF-IDF matrix shape: (5572, 6321)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 41076 stored elements and shape (5572, 6321)>
  Coords	Values
  (0, 2845)	0.3742052644305445
  (0, 4110)	0.25551526608016073
  (0, 1204)	0.2870925471092463
  (0, 390)	0.27989333397765176
  (0, 750)	0.31613011954967907
  (0, 2264)	0.20594949396141035
  (0, 6184)	0.250855761537219
  (0, 748)	0.3572193734409173
  (0, 982)	0.31613011954967907
  (0, 2160)	0.13171426300074174
  (0, 189)	0.3742052644305445
  (0, 6006)	0.20908309343990275
  (1, 2967)	0.4151280735978849
  (1, 2816)	0.5711999523791034
  (1, 6104)	0.4388200065478342
  (1, 3796)	0.5557304191479696
  (2, 2040)	0.15373353356620328
  (2, 1712)	0.4753622620486791
  (2, 6149)	0.25500701794657454
  (2, 1079)	0.2602821879263758
  (2, 6114)	0.19284077973640795
  (2, 1255)	0.2699538017092184
  (2, 1922)	0.24629294642972502
  (2, 5591)	0.29550549732084663
  (2, 3293)	0.2113693403490519
  :	:
  (5567, 5712)	0.2637099015368332
  (5567, 1135)	0.2850

# Train and test split

In [ ]:
#the dataset is imbalanced, so we use stratified splitting to preserve the original class distribution in both
#the training and the test sets

train_df, test_df = train_test_split(
    df,
    test_size=0.3,    #we chose 30% for testing
    random_state=42,
    stratify=df["label"]
)

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)
print("\nTest label distribution:\n", test_df["label"].value_counts())


Train size: (3900, 3)
Test size: (1672, 3)

Test label distribution:
 label
ham     1448
spam     224
Name: count, dtype: int64


# Labeled and unlabeled split

In [ ]:
#we simulate a semi-supervised learning scenario by hiding most of the labels in the training set

labeled_df, unlabeled_df = train_test_split(
    train_df,
    test_size=0.7,              #for that, 70% of training data becomes unlabeled
    random_state=42,
    stratify=train_df["label"]
)

#we mark unlabeled examples with -1 to indicate that they are unlabeled
unlabeled_df = unlabeled_df.copy()
unlabeled_df["label_ssl"] = -1

print("Labeled set size:", labeled_df.shape)
print("Unlabeled set size:", unlabeled_df.shape)

Labeled set size: (1170, 3)
Unlabeled set size: (2730, 4)


# Applying TF-IDF to train, unlabeled, and test Sets

In [ ]:
#we apply the same TF-IDF representation to each data split

#we transform the labeled texts into TF-IDF vectors
X_labeled = vectorizer.transform(labeled_df["clean_text"])
#we store the true labels for the labeled data
y_labeled = labeled_df["label"]

#we transform the unlabeled texts using the same TF-IDF
X_unlabeled = vectorizer.transform(unlabeled_df["clean_text"])
#here we store the SSL labels
y_unlabeled = unlabeled_df["label_ssl"]   #all values are -1, unlabeled

#we also transform the test texts into TF-IDF
X_test = vectorizer.transform(test_df["clean_text"])
#we store the true labels of the test set
y_test = test_df["label"]

print("X_labeled shape:", X_labeled.shape)
print("X_unlabeled shape:", X_unlabeled.shape)
print("X_test shape:", X_test.shape)


X_labeled shape: (1170, 6321)
X_unlabeled shape: (2730, 6321)
X_test shape: (1672, 6321)


# Baseline supervised model, Logistic Regression

In [ ]:
#we create a logistic regression classifier
#this model will learn from labeled examples only
baseline_model = LogisticRegression(max_iter=2000)

#we train the model using TF-IDF features and true labels
baseline_model.fit(X_labeled, y_labeled)

#we predict labels for the test set
baseline_pred = baseline_model.predict(X_test)

#we compute overall accuracy of the model
print("Baseline accuracy:", accuracy_score(y_test, baseline_pred))

#below we show detailed evaluation metrics
#precision, recall and F1 score for each class
print("\nClassification report (Baseline):")
print(classification_report(y_test, baseline_pred))



Baseline accuracy: 0.9144736842105263

Classification report (Baseline):
              precision    recall  f1-score   support

         ham       0.91      1.00      0.95      1448
        spam       0.96      0.38      0.54       224

    accuracy                           0.91      1672
   macro avg       0.93      0.69      0.75      1672
weighted avg       0.92      0.91      0.90      1672



# Self Training model

In [ ]:
from scipy.sparse import vstack

#the self training model uses the logistic regression as the base classifier
#this model will use labeled data and confident predictions on unlabeled data
base_classifier = LogisticRegression(max_iter=2000)

#we create the self training classifier
#as we commented before, unlabeled samples are marked with -1
self_training_model = SelfTrainingClassifier(
    estimator=base_classifier,
    threshold=0.9        #we set as a threshold 0.9, which means only high confidence predictions are accepted
)

#we combine labeled and unlabeled data
X_ssl = vstack([X_labeled, X_unlabeled])
y_ssl = np.concatenate([y_labeled, y_unlabeled])

#we train the self training model
self_training_model.fit(X_ssl, y_ssl)

#we predict labels for the test set
ssl_pred = self_training_model.predict(X_test)

#we evaluate the self training model
print("Self-training accuracy:", accuracy_score(y_test, ssl_pred))
print("\nClassification report (Self-training):")
print(classification_report(y_test, ssl_pred))


Self-training accuracy: 0.8911483253588517

Classification report (Self-training):
              precision    recall  f1-score   support

         ham       0.89      1.00      0.94      1448
        spam       0.96      0.20      0.33       224

    accuracy                           0.89      1672
   macro avg       0.92      0.60      0.63      1672
weighted avg       0.90      0.89      0.86      1672



### Note: The self training model achieves high overall accuracy and very high precision for the spam class. However, the recall for spam is low, meaning that many spam messages are not detected. Which is expected due to the high threshold and the class imbalance, that also led to a strong performance on the ham class.

# Comparison and conclusion

In [ ]:
#we are going to make a comparison between the baseline accuracy and the self training to see the differences
print("Baseline model vs Self-training model")
print("-------------------------------------")
print("Baseline accuracy:", accuracy_score(y_test, baseline_pred))
print("Self-training accuracy:", accuracy_score(y_test, ssl_pred))

Baseline model vs Self-training model
-------------------------------------
Baseline accuracy: 0.9144736842105263
Self-training accuracy: 0.8911483253588517


## Conclusion: As we can see, the baseline model has high accuracy while the self training model is lower. It detects spam only when it is very confident, so precision is high but recall is low. And the reason why is because of the high threshold that we set, and the class imbalance as I commented before.

# Clustering + Labeling

### Now we are going to apply another semi-supervised learning method to the same dataset. First, we import the necessary libraries for the clustering + labeling approach.




In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
from scipy.sparse import vstack

### Here we combine labeled and unlabeled TF-IDF vectors (we use the same training split as before)

In [ ]:
X_ssl = vstack([X_labeled, X_unlabeled])

We fit a clustering model on all our training data and we use 2 clusters because we have 2 classes: ham and spam.


In [ ]:
kmeans = MiniBatchKMeans(n_clusters=2, random_state=42)
kmeans.fit(X_ssl)

MiniBatchKMeans(n_clusters=2, random_state=42)

### Here we predict which cluster each labeled sample belongs to, so we can assign a class label to each cluster.


In [ ]:
labeled_cluster_ids = kmeans.predict(X_labeled)


###  We assign a class label to each cluster using majority vote


In [ ]:
#first, we will assign a real class (ham or spam) to each cluster
#to do this, we look at the labeled samples inside each cluster and choose the majority class

cluster_to_label = {}  #we create a dictionary to store cluster -> class label

#in this case, we have 2 clusters: 0 and 1
for c in range(2):

    #we search the positions of labeled samples that belong to cluster c
    idx = np.where(labeled_cluster_ids == c)[0]

    #if no labeled samples fall in this cluster,we assign the most common class in the labeled dataset
    #that's usually rare, but can happen
    if len(idx) == 0:
        cluster_to_label[c] = y_labeled.value_counts().idxmax()

    else:
        #we look at the real labels of those samples
        labels_in_cluster = y_labeled.iloc[idx]

        #we find the most frequent label (ham or spam)
        majority_label = labels_in_cluster.value_counts().idxmax()

        #and we assign that label to the cluster
        cluster_to_label[c] = majority_label

#visualization of the final mapping
print("Cluster to label mapping:", cluster_to_label)

Cluster to label mapping: {0: 'ham', 1: 'spam'}


### As we can observe, the clustering separated the data into two groups: cluster 0 mainly contains ham and cluster 1 mainly contains spam.


In [ ]:
#now we assign pseudo labels to all training samples (labeled and unlabeled)
train_cluster_ids = kmeans.predict(X_ssl)
y_pseudo_all = np.array([cluster_to_label[c] for c in train_cluster_ids])

print("Pseudo label distribution:", dict(zip(*np.unique(y_pseudo_all, return_counts=True))))

Pseudo label distribution: {np.str_('ham'): np.int64(3398), np.str_('spam'): np.int64(502)}


### The pseudo label distribution shows that most samples were assigned to ham, which reflects the class imbalance of the dataset that was commented before

In [ ]:
#we train a supervised classifier on the pseudo labeled training set
#now we treat the pseudo labels as if they were real labels
cluster_label_model = LogisticRegression(max_iter=2000)
cluster_label_model.fit(X_ssl, y_pseudo_all)

#we finally evaluate on the real test set
cluster_pred = cluster_label_model.predict(X_test)

print("Clustering+Labeling accuracy:", accuracy_score(y_test, cluster_pred))
print("\nClassification report (Clustering+Labeling):")
print(classification_report(y_test, cluster_pred))

Clustering+Labeling accuracy: 0.9216507177033493

Classification report (Clustering+Labeling):
              precision    recall  f1-score   support

         ham       0.93      0.98      0.96      1448
        spam       0.83      0.53      0.64       224

    accuracy                           0.92      1672
   macro avg       0.88      0.75      0.80      1672
weighted avg       0.92      0.92      0.91      1672



In [ ]:
#now were doing a quick comparison with the previous models
print("Baseline accuracy:", accuracy_score(y_test, baseline_pred))
print("Self-training accuracy:", accuracy_score(y_test, ssl_pred))
print("Clustering+Labeling accuracy:", accuracy_score(y_test, cluster_pred))

Baseline accuracy: 0.9144736842105263
Self-training accuracy: 0.8911483253588517
Clustering+Labeling accuracy: 0.9216507177033493


### Note: The clustering and labeling approach achieves slightly higher accuracy than the baseline model. It significantly improves the recall for the spam class compared to self training. It achieves 53 % for the recall compared to the 20 % recall of self training, which is a great improvement.We clearly achieve a better balance between precision and recall.


### Final conclusion: The clustering and labeling approach achieved the best overall performance. It improved spam detection compared to self training and slightly outperformed the baseline. This suggests that using unlabeled data can help improve classification results...